In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

In [ ]:
np.random.seed(42)

Wygeneruj n obserwacji z rozkładu $N(\theta, \sigma^2)$


a) $\theta$ = 0, $\sigma$ = 1,

b) $\theta$ = 0, $\sigma$ = 2,

c) $\theta$ = 4, $\sigma$ = 1,

d) $\theta$ = 4, $\sigma$ = 2.


In [ ]:
def generate_sample(theta, sigma, n=50):
    return theta + sigma * np.random.randn(n)

Na tej postawie oblicz wartości estymatora $\theta$

In [ ]:
def est_1(data): # średnia
    return np.mean(data)

In [ ]:
def est_2(data): # mediana
    return np.median(data)

In [ ]:
max_n = 100 # maksymalna wartośc n
weights_3 = np.random.uniform(0, 1, max_n) # globalne wagi dla est_3

In [ ]:
def est_3(data): # ważona średnia, z własnymi wagami
    weights = weights_3[0:len(data)]
    weights = weights / weights.sum()
    return np.dot(data, weights)

In [ ]:
def est_4(data, eps=1e-12): 
    n = len(data)
    data = np.sort(data)  # lepiej użyć sortowania NumPy
    # kwantyle od 0 do 1, przycięte eps
    i = np.linspace(0, 1, n+1)
    i = np.clip(i, eps, 1-eps)
    # gęstości standardowego rozkładu normalnego
    phi_vals = norm.pdf(norm.ppf(i))
    # wagi: phi(Phi^-1((i-1)/n)) - phi(Phi^-1(i/n))
    weights = phi_vals[1:] - phi_vals[:-1]
    return np.dot(data, weights)

In [ ]:
def est_5(data):
    data = np.asarray(data)
    data = data[data != 0]  # ignorujemy zera
    return len(data) / np.sum(1.0 / data)

In [ ]:
def est_6(data):
    data = np.asarray(data)
    mean_cubed = np.mean(data ** 3)
    return np.sign(mean_cubed) * np.abs(mean_cubed) ** (1/3)

In [ ]:
def est_7(data):
    data = np.asarray(data)
    return 0.5 * (np.min(data) + np.max(data))

In [ ]:
estimators = [est_1, est_2, est_3, est_4, est_5, est_6, est_7]

In [ ]:
def test_case(theta, sigma, num_samples, sample_size=100):
    results = np.zeros((len(estimators), num_samples))

    for j in range(num_samples):
        sample = generate_sample(theta, sigma, sample_size)
        for i, est in enumerate(estimators):
            results[i, j] = est(sample)

    return results

In [ ]:
thetas = [0, 0, 4, 4]
sigmas = [1, 2, 1, 2]

In [ ]:
sample_sizes = [20, 50, 100]

num_samples = 100**2

results_stats = {}
results = {}

In [ ]:
def theta_stats(estimates, true_theta, true_sigma):
    mean_est = np.mean(estimates)
    var_est = np.var(estimates, ddof=1)
    bias = mean_est - true_theta
    mse = np.mean((estimates - true_theta)**2)
    return mean_est, var_est, bias, mse

In [ ]:
for n in sample_sizes:
    stats_for_n = []
    results_for_n = []
    for theta, sigma in zip(thetas, sigmas):
        estimates = test_case(theta, sigma, num_samples, n)
        results_for_n.append(estimates)

        stats = [theta_stats(estimate, theta, sigma) for estimate in estimates]
        stats_for_n.append(stats)
    results[n] = results_for_n
    results_stats[n] = stats_for_n

Wykresy pudełkowe dla pierwszego zestawu parametrów (theta=0, sigma=1)

dla ustalonego n (num_samples), a (test_case), e (esymatora)

results_stats[n][a..d][e] to dokładnie jego (mean, varm, bias, mse) 

In [ ]:
import os
import pandas as pd

os.makedirs("plots", exist_ok=True)

for n in sample_sizes:
    for param_idx, (theta, sigma) in enumerate(zip(thetas, sigmas)):
        data_to_plot = [results[n][param_idx][i] for i in range(len(estimators))]

        plt.figure(figsize=(12, 6))
        plt.boxplot(
            data_to_plot,
            tick_labels=[f"est_{i+1}" for i in range(len(estimators))],
            flierprops=dict(marker='o', color='red', alpha=0.5)  # outliery jako czerwone kółka
        )
        plt.title(f"Porównanie estymatorów (n={n}, theta={theta}, sigma={sigma})")
        plt.ylabel("Wartości estymatora")
        plt.xlabel("Estymatory")
        plt.grid(True)

        # wyznaczamy zakres Y większy niż standardowe wąsy
        all_data = np.hstack(data_to_plot)
        q1 = np.percentile(all_data, 25)
        q3 = np.percentile(all_data, 75)
        iqr = q3 - q1

        # zwiększamy zakres w porównaniu do standardowych 1.5*IQR, dla lepszej wydocznosci
        delta = 5
        plt.ylim(-delta + q1 - iqr, q3 + iqr + delta)

        # zapis PNG
        png_filename = f"plots/boxplot_theta{theta}_sigma{sigma}_n{n}.png"
        plt.savefig(png_filename)
        plt.close()

        # Statystyki w pandas
        stats_dict = {
            "Estimator": [],
            "Mean": [],
            "Variance": [],
            "Bias": [],
            "MSE": []
        }

        for i, est in enumerate(estimators):
            mean_est, var_est, bias, mse = results_stats[n][param_idx][i]
            stats_dict["Estimator"].append(f"est_{i+1}")
            stats_dict["Mean"].append(mean_est)
            stats_dict["Variance"].append(var_est)
            stats_dict["Bias"].append(bias)
            stats_dict["MSE"].append(mse)

        df_stats = pd.DataFrame(stats_dict)

        # zaokrąglamy do 2 miejsc po przecinku
        df_stats = df_stats.round(2)

        # zapis do CSV przez pandas
        csv_filename = f"plots/stats_theta{theta}_sigma{sigma}_n{n}.csv"
        df_stats.to_csv(csv_filename, index=False)

        print(f"Zapisano wykres: {png_filename} i statystyki: {csv_filename}")

Zapisano wykres: plots/boxplot_theta0_sigma1_n20.png i statystyki: plots/stats_theta0_sigma1_n20.csv
Zapisano wykres: plots/boxplot_theta0_sigma2_n20.png i statystyki: plots/stats_theta0_sigma2_n20.csv
Zapisano wykres: plots/boxplot_theta4_sigma1_n20.png i statystyki: plots/stats_theta4_sigma1_n20.csv
Zapisano wykres: plots/boxplot_theta4_sigma2_n20.png i statystyki: plots/stats_theta4_sigma2_n20.csv
Zapisano wykres: plots/boxplot_theta0_sigma1_n50.png i statystyki: plots/stats_theta0_sigma1_n50.csv
Zapisano wykres: plots/boxplot_theta0_sigma2_n50.png i statystyki: plots/stats_theta0_sigma2_n50.csv
Zapisano wykres: plots/boxplot_theta4_sigma1_n50.png i statystyki: plots/stats_theta4_sigma1_n50.csv
Zapisano wykres: plots/boxplot_theta4_sigma2_n50.png i statystyki: plots/stats_theta4_sigma2_n50.csv
Zapisano wykres: plots/boxplot_theta0_sigma1_n100.png i statystyki: plots/stats_theta0_sigma1_n100.csv
Zapisano wykres: plots/boxplot_theta0_sigma2_n100.png i statystyki: plots/stats_theta0_si